# Laboratorio 7 — Comparación de Modelos de Clasificación
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Entrenar y comparar: **Decision Tree**, **Naive Bayes**, **SVM** y **KNN**
- Seleccionar el mejor modelo usando validación cruzada
- Reflexionar sobre el trade-off complejidad/interpretabilidad

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

# Preparar datos Titanic
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked", drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked_Q", "Embarked_S"]
X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Datos preparados.")

## 1. Definir modelos

In [ ]:
dt  = DecisionTreeClassifier(max_depth=4, random_state=42)
nb  = GaussianNB()
svm = SVC(kernel="rbf", C=1.0, random_state=42, probability=True)
knn = KNeighborsClassifier(n_neighbors=5)

modelos = {
    "Decision Tree":  (dt,  X_train, X_test),
    "Naive Bayes":    (nb,  X_train_scaled, X_test_scaled),
    "SVM":            (svm, X_train_scaled, X_test_scaled),
    "KNN":            (knn, X_train_scaled, X_test_scaled),
}

## 2. Entrenar y comparar

In [ ]:
resultados = {}
for nombre, (modelo, X_tr, X_te) in modelos.items():
    modelo.fit(X_tr, y_train)
    y_pred = modelo.predict(X_te)
    y_prob = modelo.predict_proba(X_te)[:, 1]
    resultados[nombre] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "AUC":      roc_auc_score(y_test, y_prob),
    }

resultados_df = pd.DataFrame(resultados).T
print(resultados_df.round(4))

## 3. Validación cruzada comparativa

In [ ]:
for nombre, (modelo, _, _) in modelos.items():
    if nombre in ["SVM", "KNN", "Naive Bayes"]:
        pipe = Pipeline([("scaler", StandardScaler()), ("modelo", modelo)])
    else:
        pipe = modelo
    scores = cross_val_score(pipe, X, y, cv=5, scoring="f1")
    print(f"{nombre:20s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

## 4. Visualización comparativa

In [ ]:
resultados_df.plot(kind="bar", figsize=(10, 6), edgecolor="black")
plt.title("Comparación de modelos — Titanic")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.ylim(0.5, 1.0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. Curvas ROC comparativas

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for nombre, (modelo, X_tr, X_te) in modelos.items():
    y_prob = modelo.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f"{nombre} (AUC={auc:.3f})")

plt.plot([0,1],[0,1],"k--", label="Aleatorio")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curvas ROC — Comparación de modelos")
plt.legend()
plt.grid(True)
plt.show()

## 6. Efecto de K en KNN

In [ ]:
ks = range(1, 21)
scores_k = []
for k in ks:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(
        Pipeline([("scaler", StandardScaler()), ("knn", knn_k)]),
        X, y, cv=5, scoring="accuracy"
    ).mean()
    scores_k.append(score)

plt.plot(ks, scores_k, marker="o", color="steelblue")
plt.title("Accuracy de KNN según K")
plt.xlabel("K")
plt.ylabel("Accuracy (CV)")
plt.xticks(ks)
plt.grid(True)
plt.show()

## Ejercicios

### Ejercicio 1 — Ajustar C del SVM

In [ ]:
for C in [0.01, 0.1, 1, 10, 100]:
    svm_c = SVC(kernel="rbf", C=C, probability=True, random_state=42)
    svm_c.fit(X_train_scaled, y_train)
    yp = svm_c.predict(X_test_scaled)
    yprob = svm_c.predict_proba(X_test_scaled)[:, 1]
    print(f"C={C:6}: Accuracy={accuracy_score(y_test,yp):.4f}, AUC={roc_auc_score(y_test,yprob):.4f}")

### Desafío — Función comparar_modelos

In [ ]:
def comparar_modelos(modelos_dict, X, y, cv=5):
    resultados = {}
    for nombre, modelo in modelos_dict.items():
        scores = cross_val_score(modelo, X, y, cv=cv, scoring="f1")
        resultados[nombre] = {"F1_mean": scores.mean(), "F1_std": scores.std()}
    return pd.DataFrame(resultados).T.sort_values("F1_mean", ascending=False)

modelos_simples = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Naive Bayes":   GaussianNB(),
}
print(comparar_modelos(modelos_simples, X, y))